In [1]:
!pip install -q diffusers transformers accelerate safetensors ipywidgets pillow torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 35.8 MB/s eta 0:00:00


In [3]:
import torch
from diffusers import StableDiffusionPipeline
import ipywidgets as widgets
from IPython.display import display, clear_output

# Проверка устройства (использовать CUDA если доступно)
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "runwayml/stable-diffusion-v1-5"

print(f"Используемое устройство: {device}")

# Загрузка предобученной модели
if torch.cuda.is_available():
    # Используем float16 для ускорения и экономии памяти на GPU
    pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
else:
    pipe = StableDiffusionPipeline.from_pretrained(model_id)

pipe = pipe.to(device)
print("Модель успешно загружена!")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Используемое устройство: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель успешно загружена!


In [4]:
# --- Создание элементов управления (Виджеты) ---
style = {'description_width': 'initial'}

prompt = widgets.Textarea(
    value="красивый закат над морем, 4k, детализация",
    placeholder='Введите описание на английском...',
    description="Промпт:",
    layout={'width': '100%', 'height': '80px'}
)

negative = widgets.Text(
    value="bad quality, blurry, low resolution",
    description="Негативный промпт:",
    style=style, layout={'width': '100%'}
)

steps = widgets.IntSlider(value=30, min=5, max=100, step=1, description="Шаги (Steps):", style=style)
guidance = widgets.FloatSlider(value=7.5, min=1.0, max=20.0, step=0.5, description="Креативность (CFG):", style=style)
width = widgets.IntSlider(value=512, min=128, max=1024, step=64, description="Ширина:", style=style)
height = widgets.IntSlider(value=512, min=128, max=1024, step=64, description="Высота:", style=style)
seed = widgets.IntText(value=0, description="Seed (0 = рандом):", style=style)

run_btn = widgets.Button(
    description="🚀 Сгенерировать",
    button_style='success', # 'success', 'info', 'warning', 'danger' or ''
    layout={'width': '100%', 'height': '40px'}
)

out = widgets.Output()

# --- Функция обработки нажатия ---
def on_click(_):
    with out:
        clear_output(wait=True)
        print("🎨 Генерация изображения... Пожалуйста, подождите.")

        # Настройка Seed
        if int(seed.value) != 0:
            gen = torch.Generator(device=device).manual_seed(int(seed.value))
        else:
            gen = None

        try:
            image = pipe(
                prompt.value,
                negative_prompt=negative.value if negative.value else None,
                num_inference_steps=int(steps.value),
                guidance_scale=float(guidance.value),
                height=int(height.value),
                width=int(width.value),
                generator=gen
            ).images[0]

            clear_output(wait=True)
            display(image)
        except Exception as e:
            print(f"❌ Ошибка: {e}")

run_btn.on_click(on_click)

# --- Отображение интерфейса ---
ui = widgets.VBox([
    widgets.HTML("<h2>Stable Diffusion Interface</h2>"),
    prompt, negative,
    widgets.HBox([steps, guidance]),
    widgets.HBox([width, height]),
    seed, run_btn, out
])

display(ui)